# 12 OpenOximetry TinyKAN-Inspired KAN Comparison

## Scope and Outputs

This notebook evaluates a compact TinyKAN-inspired B-spline KAN binary classifier on the OpenOximetry canonical feature table. It is **not** a reproduction of the full TinyKAN-HAR zero-shot architecture. The HAR-specific semantic projection, activity embeddings, cosine zero-shot scoring, and UCI HAR assumptions are intentionally omitted because this project uses a binary low-oxygen-risk target.

Contract constraints:
- Data: `outputs/openox_canonical_window_features.csv`
- Target: `label` (window median SpO2 < 90)
- Features: the same 14 engineered OpenOximetry columns used by notebook 11
- Split: `GroupKFold(n_splits=5)` grouped by `recording_id`
- Scaling: fit `StandardScaler` on each training fold only
- Metrics: accuracy, precision, recall, F1, confusion matrix, and model parameter count

Outputs:
- `outputs/openox_tinykan_summary.csv`
- `outputs/openox_tinykan_split_diagnostics.csv`
- `outputs/openox_tinykan_predictions.csv`
- `outputs/openox_tinykan_feature_attributions.csv`


## Data Preparation

Load the canonical OpenOximetry feature table, validate the fixed schema, and prepare grouped folds by recording. SpO2-derived fields remain label evidence only and are not used as model inputs.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

SEED = 42
N_SPLITS = 5
N_EPOCHS = 80
LEARNING_RATE = 0.01
WEIGHT_DECAY = 1e-4
KAN_DIMS = (14, 12, 1)
N_SPLINE_KNOTS = 8
SPLINE_DEGREE = 3
SPLINE_RANGE = (-3.0, 3.0)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

FEATURE_COLUMN_NAMES = (
    "mean", "std", "minimum", "maximum", "value_range", "rms", "skewness", "kurtosis",
    "energy", "derivative_std", "dominant_frequency_hz", "bandpower_0_5_2_hz",
    "bandpower_2_5_hz", "bandpower_ratio_0_5_2_to_2_5",
)
REQUIRED_COLUMNS = {"recording_id", "label", *FEATURE_COLUMN_NAMES}


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "outputs").is_dir():
            return candidate
    raise RuntimeError("Could not find project root containing notebooks/ and outputs/")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FEATURE_CSV = OUTPUT_DIR / "openox_canonical_window_features.csv"

feature_df = pd.read_csv(FEATURE_CSV)
missing = sorted(REQUIRED_COLUMNS.difference(feature_df.columns))
assert not missing, f"Missing required columns: {missing}"

for col in FEATURE_COLUMN_NAMES:
    feature_df[col] = pd.to_numeric(feature_df[col], errors="raise")
feature_df["label"] = pd.to_numeric(feature_df["label"], errors="raise").astype(int)
feature_df["recording_id"] = feature_df["recording_id"].astype(str)

X = feature_df.loc[:, FEATURE_COLUMN_NAMES].to_numpy(dtype=np.float32)
y = feature_df["label"].to_numpy(dtype=np.int64)
groups = feature_df["recording_id"].to_numpy()

print(f"Feature source: {FEATURE_CSV}")
print(f"Rows: {len(feature_df)}")
print(f"Recordings: {feature_df['recording_id'].nunique()}")
print(f"Positive label rate: {y.mean():.4f}")

## Compact B-Spline KAN Model

The model adapts the B-spline KAN layer pattern from the TinyKAN reproduction, but changes the head and task to binary OpenOximetry classification. The architecture is deliberately small: 14 inputs, 8 hidden KAN units, and 1 binary logit.

In [ ]:
class BSpline(nn.Module):
    def __init__(self, n_knots=8, degree=3, range_min=-2.0, range_max=2.0):
        super().__init__()
        self.n_knots = n_knots
        self.degree = degree
        self.range_min = range_min
        self.range_max = range_max
        self.register_buffer("centers", torch.linspace(range_min, range_max, n_knots))
        self.register_buffer("step", torch.tensor((range_max - range_min) / (n_knots - 1)))

    def forward(self, u):
        if self.degree != 3:
            raise ValueError("This compact notebook implementation supports cubic B-splines only.")
        u_clamped = u.clamp(self.range_min, self.range_max)
        distance = torch.abs((u_clamped.unsqueeze(-1) - self.centers) / (self.step + 1e-10))
        inner = (2.0 / 3.0) - distance.pow(2) + 0.5 * distance.pow(3)
        outer = (2.0 - distance).clamp(min=0).pow(3) / 6.0
        return torch.where(distance < 1.0, inner, outer)


class KANLayer(nn.Module):
    def __init__(self, d_in, d_out, n_knots=8, degree=3, range_min=-2.0, range_max=2.0):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.n_knots = n_knots
        self.W = nn.Parameter(torch.empty(d_out, d_in))
        self.b = nn.Parameter(torch.zeros(d_out))
        nn.init.xavier_uniform_(self.W)
        self.spline = BSpline(n_knots, degree, range_min, range_max)
        self.theta = nn.Parameter(torch.empty(d_out, n_knots))
        self._init_spline_to_identity()

    def _init_spline_to_identity(self):
        grid = torch.linspace(self.spline.range_min, self.spline.range_max, 100)
        basis_grid = self.spline(grid)
        target = grid.unsqueeze(1)
        theta_init = torch.linalg.lstsq(basis_grid, target).solution.squeeze(-1)
        self.theta.data = theta_init.unsqueeze(0).expand(self.d_out, -1).clone()

    def forward(self, x):
        u = F.linear(x, self.W, self.b)
        batch = u.shape[0]
        basis = self.spline(u.reshape(-1)).reshape(batch, self.d_out, self.n_knots)
        return (basis * self.theta.unsqueeze(0)).sum(dim=-1)

    def spline_smoothness_penalty(self):
        diff2 = self.theta[:, 2:] - 2 * self.theta[:, 1:-1] + self.theta[:, :-2]
        return diff2.pow(2).sum()


class KANBinaryClassifier(nn.Module):
    def __init__(self, dims=(14, 8, 1), n_knots=8, degree=3, spline_range=(-2.0, 2.0)):
        super().__init__()
        self.layers = nn.ModuleList(
            KANLayer(dims[i], dims[i + 1], n_knots, degree, spline_range[0], spline_range[1])
            for i in range(len(dims) - 1)
        )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def regularization(self):
        return sum(layer.spline_smoothness_penalty() for layer in self.layers)


dummy_model = KANBinaryClassifier(KAN_DIMS, N_SPLINE_KNOTS, SPLINE_DEGREE, SPLINE_RANGE)
param_count = sum(p.numel() for p in dummy_model.parameters() if p.requires_grad)
print(f"KAN binary classifier size: {param_count} trainable parameters")

## Grouped Evaluation

Train and evaluate the compact KAN model on five recording-disjoint folds. Per-fold scalers are fit on training windows only. Feature attributions use gradient-times-input on standardized test features.

In [ ]:
def train_fold_model(x_train, y_train):
    torch.manual_seed(SEED)
    model = KANBinaryClassifier(KAN_DIMS, N_SPLINE_KNOTS, SPLINE_DEGREE, SPLINE_RANGE)
    pos_weight_value = (len(y_train) - y_train.sum()) / max(1, y_train.sum())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32))
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    x_train_t = torch.tensor(x_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

    model.train()
    for _ in range(N_EPOCHS):
        optimizer.zero_grad()
        logits = model(x_train_t)
        loss = criterion(logits, y_train_t) + 1e-5 * model.regularization()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
    return model


def predict_with_attributions(model, x_test):
    model.eval()
    x_test_t = torch.tensor(x_test, dtype=torch.float32, requires_grad=True)
    logits = model(x_test_t)
    probs = torch.sigmoid(logits).detach().numpy().ravel()
    preds = (probs >= 0.30).astype(int)
    logits.sum().backward()
    attributions = (x_test_t.grad * x_test_t).detach().numpy()
    return probs, preds, attributions


split_rows = []
prediction_rows = []
attribution_arrays = []
pooled_true = []
pooled_pred = []
pooled_prob = []

splitter = GroupKFold(n_splits=N_SPLITS)
for split_index, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups=groups), start=1):
    scaler = StandardScaler()
    x_train = scaler.fit_transform(X[train_idx]).astype(np.float32)
    x_test = scaler.transform(X[test_idx]).astype(np.float32)
    y_train = y[train_idx]
    y_test = y[test_idx]

    model = train_fold_model(x_train, y_train)
    probs, preds, attributions = predict_with_attributions(model, x_test)

    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()
    split_rows.append(
        {
            "split_index": split_index,
            "train_row_count": int(len(train_idx)),
            "test_row_count": int(len(test_idx)),
            "train_group_count": int(len(set(groups[train_idx].tolist()))),
            "test_group_count": int(len(set(groups[test_idx].tolist()))),
            "group_overlap_count": int(len(set(groups[train_idx].tolist()).intersection(set(groups[test_idx].tolist())))),
            "train_positive_rows": int(y_train.sum()),
            "test_positive_rows": int(y_test.sum()),
            "accuracy": float(accuracy_score(y_test, preds)),
            "precision": float(precision_score(y_test, preds, zero_division=0)),
            "recall": float(recall_score(y_test, preds, zero_division=0)),
            "f1": float(f1_score(y_test, preds, zero_division=0)),
            "true_negative": int(tn),
            "false_positive": int(fp),
            "false_negative": int(fn),
            "true_positive": int(tp),
        }
    )

    top_feature_idx = np.argmax(np.abs(attributions), axis=1)
    for local_pos, row_idx in enumerate(test_idx):
        prediction_rows.append(
            {
                "row_index": int(row_idx),
                "recording_id": groups[row_idx],
                "split_index": split_index,
                "y_true": int(y_test[local_pos]),
                "y_prob": float(probs[local_pos]),
                "y_pred": int(preds[local_pos]),
                "top_attribution_feature": FEATURE_COLUMN_NAMES[int(top_feature_idx[local_pos])],
            }
        )

    pooled_true.extend(y_test.tolist())
    pooled_pred.extend(preds.tolist())
    pooled_prob.extend(probs.tolist())
    attribution_arrays.append(attributions)

    print(
        f"Fold {split_index}: recall={recall_score(y_test, preds, zero_division=0):.4f}, "
        f"f1={f1_score(y_test, preds, zero_division=0):.4f}, "
        f"accuracy={accuracy_score(y_test, preds):.4f}"
    )

pooled_true = np.asarray(pooled_true)
pooled_pred = np.asarray(pooled_pred)
pooled_prob = np.asarray(pooled_prob)
pooled_attributions = np.vstack(attribution_arrays)
split_diagnostics_df = pd.DataFrame(split_rows)
predictions_df = pd.DataFrame(prediction_rows)

## Pooled Metrics and Attribution Summary

Summarize pooled test-set performance and aggregate the feature attribution magnitudes across all held-out windows.

In [ ]:
pooled_tn, pooled_fp, pooled_fn, pooled_tp = confusion_matrix(pooled_true, pooled_pred, labels=[0, 1]).ravel()
summary_df = pd.DataFrame(
    [
        {
            "model_name": "openox_tinykan_bspline_binary",
            "label_variant": "label",
            "fold_count": N_SPLITS,
            "parameter_count": int(param_count),
            "kan_dims": str(KAN_DIMS),
            "n_spline_knots": N_SPLINE_KNOTS,
            "spline_degree": SPLINE_DEGREE,
            "pooled_test_rows": int(len(pooled_true)),
            "pooled_test_recordings": int(feature_df["recording_id"].nunique()),
            "pooled_true_negative": int(pooled_tn),
            "pooled_false_positive": int(pooled_fp),
            "pooled_false_negative": int(pooled_fn),
            "pooled_true_positive": int(pooled_tp),
            "pooled_accuracy": float(accuracy_score(pooled_true, pooled_pred)),
            "pooled_precision": float(precision_score(pooled_true, pooled_pred, zero_division=0)),
            "pooled_recall": float(recall_score(pooled_true, pooled_pred, zero_division=0)),
            "pooled_f1": float(f1_score(pooled_true, pooled_pred, zero_division=0)),
            "mean_split_accuracy": float(split_diagnostics_df["accuracy"].mean()),
            "mean_split_precision": float(split_diagnostics_df["precision"].mean()),
            "mean_split_recall": float(split_diagnostics_df["recall"].mean()),
            "mean_split_f1": float(split_diagnostics_df["f1"].mean()),
        }
    ]
)

top_feature_counts = predictions_df["top_attribution_feature"].value_counts().to_dict()
attribution_rows = []
for feature_index, feature_name in enumerate(FEATURE_COLUMN_NAMES):
    values = pooled_attributions[:, feature_index]
    attribution_rows.append(
        {
            "feature_name": feature_name,
            "mean_signed_attribution": float(values.mean()),
            "mean_abs_attribution": float(np.abs(values).mean()),
            "std_attribution": float(values.std(ddof=0)),
            "top_driver_count": int(top_feature_counts.get(feature_name, 0)),
            "top_driver_fraction": float(top_feature_counts.get(feature_name, 0) / len(predictions_df)),
        }
    )
attribution_summary_df = pd.DataFrame(attribution_rows).sort_values("mean_abs_attribution", ascending=False).reset_index(drop=True)

summary_df.to_csv(OUTPUT_DIR / "openox_tinykan_summary.csv", index=False)
split_diagnostics_df.to_csv(OUTPUT_DIR / "openox_tinykan_split_diagnostics.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "openox_tinykan_predictions.csv", index=False)
attribution_summary_df.to_csv(OUTPUT_DIR / "openox_tinykan_feature_attributions.csv", index=False)

print("Pooled TinyKAN-inspired KAN summary:")
print(summary_df.round(4).to_string(index=False))
print("\nSplit diagnostics:")
print(split_diagnostics_df.round(4).to_string(index=False))
print("\nTop attribution features:")
print(attribution_summary_df.head(10).round(4).to_string(index=False))
print("\nSaved TinyKAN comparison outputs to outputs/openox_tinykan_*.csv")

## Interpretation Note

This notebook should be described as a TinyKAN-inspired B-spline KAN binary classifier for OpenOximetry. It uses the KAN spline-layer idea, but it does not claim to reproduce TinyKAN-HAR because the OpenOximetry task has no human-activity labels, no unseen activity classes, and no semantic embedding objective. The exported feature attributions are gradient-based local explanations rather than exact additive GAM contributions.